In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, avg

In [2]:
problem_name = os.path.basename(os.path.abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 23:24:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
grades = spark.read\
.option("multiLine", "true")\
.format("json")\
.load("files/grades.json")

In [5]:
grades.show()

+--------------------+-----+----------+
|              grades| name|student_id|
+--------------------+-----+----------+
|[{85, Math}, {90,...|Alice|         1|
|[{75, Math}, {80,...|  Bob|         2|
+--------------------+-----+----------+



In [6]:
result_df = grades.withColumn("exploded_grades", explode("grades"))\
.select(col("exploded_grades.score").alias("score"), col("exploded_grades.subject").alias("subject"))\
.groupBy("subject").agg(avg("score"))

In [7]:
result_df.show()

+-------+----------+
|subject|avg(score)|
+-------+----------+
|Science|      85.0|
|   Math|      80.0|
|English|      82.5|
+-------+----------+



In [8]:
spark.stop()